In [1]:
# exponential utility function
import pandas as pd
import numpy as np
import math
import cvxpy as cp

In [72]:
# torch to compute gradient and hessian
## variable order would be: x \in R, mu \in R^d, sigma^2 (take as a whole) \in \R^d  
import torch
from torch.autograd import grad
from torch.autograd.functional import jacobian
from torch.autograd.functional import hessian
import math

dimx = 3

#x: dim is d_x + d_x + d_x
index = 0
gamma1 = 1
gamma2 = 2



x_input = torch.tensor([1, 0.5, 0.2])
mu_input = torch.tensor([0.4, 0.5, 0.3])
sigma2_input = torch.tensor([0.2, 0.2, 0.2])

jacob_x_p = []
hes_x_p = []

for index in range(dimx):
    def func_1(x, params):
        mu = params[0:dimx]
        sigma2 = params[dimx:]

        return (-gamma1 * mu[index] + gamma1 ** 2 * sigma2[index] * x[index]) * \
                torch.exp(-gamma1 * torch.inner(x, mu) + gamma1 ** 2/2 * torch.inner(sigma2, torch.square(x)))  + gamma2 * x[index]

    params = torch.cat((mu_input, sigma2_input))
    hes = hessian(func_1, (x_input, params))
    jacob = jacobian(func_1, (x_input, params))

    f_x_p = hes[0][1][index].reshape(1, -1)
    f_p_x = f_x_p.reshape(-1, 1)
    f_x_x = hes[0][0][index][index]
    f_p_p = hes[1][1]

    j_x = jacob[0][index]
    j_p = jacob[1]

    partial_x_p = -j_p / j_x
    partial_x2_p2 = -(j_x**2 * f_p_p + f_x_x * torch.mm(j_p.reshape(-1, 1), j_p.reshape(1, -1)) - j_x * (torch.mm(j_p.reshape(-1, 1), f_x_p) + torch.mm(f_p_x, j_p.reshape(1, -1))))/(j_x**3)

    jacob_x_p.append(partial_x_p)
    hes_x_p.append(partial_x2_p2)




In [79]:
import numpy as np
a = np.array([[1,2, 4],[3,4, 5]])
a_cov = np.cov(a)
np.diag(np.diagonal(a_cov))

array([[2.33333333, 0.        ],
       [0.        , 1.        ]])

In [17]:
# exponential utility function
import pandas as pd
import numpy as np
import math
import cvxpy as cp

import torch 
from torch.autograd.functional import jacobian
from torch.autograd.functional import hessian

class exp_utility_portfolio:
    def __init__(self, return_mean, return_cov, lambda_0, lambda_1, lambda_2):
        self.return_mean = return_mean
        self.return_cov = return_cov
        self.lambda_0 = lambda_0
        self.lambda_1 = lambda_1
        self.lambda_2 = lambda_2
        self.port_num = len(self.return_mean)

    def generate(self, sample_num):
        sample_data = np.random.multivariate_normal(self.return_mean, self.return_cov, size = sample_num)
        return sample_data

    def normal_idpd_pf(self, sample_data):
        mu = np.mean(sample_data, axis = 0)
        sigma2 = np.diagonal(np.cov(sample_data.T))
        return mu, sigma2
    def normal_idpd_opt(self, sample_data):

        mu, sigma2 = self.normal_idpd_pf(sample_data)

        sample_num, __ = sample_data.shape
        
        sample_data = np.random.multivariate_normal(mu, np.diag(sigma2), 100 * sample_num)
        pf_sol = self.saa(sample_data)

        return pf_sol, mu, sigma2
    
    def normal_pf_eval(self, sol, sample_data):
        mu, sigma2 = self.normal_idpd_pf(sample_data)

        sample_num, __ = sample_data.shape
        
        mc_data = np.random.multivariate_normal(mu, np.diag(sigma2), 1000 * sample_num)
        perform_eval = self.lambda_2 * np.linalg.norm(sol) ** 2
        sample_num, __ = mc_data.shape
        for i in range(sample_num):
            perform_eval += self.lambda_0 * math.exp(-self.lambda_1 * np.dot(mc_data[i], sol)) / sample_num
        return perform_eval 
        

    def normal_idpd_pf_correction(self, sol, sample_data, type = 'empirical'):
        ## empirical: base estimator is the empirical estimator
        ## parametric: base estimator is the estimator under the parametric distribution

        mu, sigma2 = self.normal_idpd_pf(sample_data)
        sample_num, __ = sample_data.shape
        # compute the covariance matrix of the asymptotic parametric distribution
        ## asymptotic variance of empirical variance estimator
        Var_p = list(sigma2) + list([np.mean([(sample_data[j][i] - mu[i])** 4 for j in range(sample_num)]) - sigma2[i]**2 for i in range(self.port_num)])

        Phi_p = torch.diag(torch.from_numpy(np.array(Var_p)))

        # compute I_h
        ## compute hes_x_p by implicit function theorem
        x_input = torch.from_numpy(sol)
        mu_input = torch.from_numpy(mu)
        sigma2_input = torch.from_numpy(sigma2)
        dimx = self.port_num

        jacob_x_p = []
        hes_x_p = []

        for index in range(dimx):
            def func_1(x, params):
                mu = params[0:dimx]
                sigma2 = params[dimx:]

                return self.lambda_0 * (-self.lambda_1 * mu[index] + self.lambda_1 ** 2 * sigma2[index] * x[index]) * \
                        torch.exp(- self.lambda_1 * torch.inner(x, mu) + self.lambda_1 ** 2/2 * torch.inner(sigma2, torch.square(x))) + 2 * self.lambda_2 * x[index]

            params = torch.cat((mu_input, sigma2_input))
            
            jacob = jacobian(func_1, (x_input, params))
            j_x = jacob[0][index]
            j_p = jacob[1]

            partial_x_p = -j_p / j_x
            jacob_x_p.append(partial_x_p)

            if type == 'parametric':
                hes = hessian(func_1, (x_input, params))
                f_x_p = hes[0][1][index].reshape(1, -1)
                f_p_x = f_x_p.reshape(-1, 1)
                f_x_x = hes[0][0][index][index]
                f_p_p = hes[1][1]

                partial_x2_p2 = -(j_x**2 * f_p_p + f_x_x * torch.mm(j_p.reshape(-1, 1), j_p.reshape(1, -1)) - j_x * (torch.mm(j_p.reshape(-1, 1), f_x_p) + torch.mm(f_p_x, j_p.reshape(1, -1))))/(j_x**3)

            
                hes_x_p.append(partial_x2_p2)
        ## compute hes_h_p
        if type == 'parametric':
            loop_num = 100 * sample_num
        else:
            loop_num = sample_num
        gradient_interact = 0
        sample_hes_h_p = 0

        for i in range(loop_num):
            if type == 'parametric':
                rv = torch.normal(mean = mu_input, std = torch.sqrt(sigma2_input))
            elif type == 'empirical':
                rv = torch.from_numpy(sample_data[i % sample_num])
            else:
                raise NotImplementedError
            
            def cost_func(x):
                return self.lambda_0 * torch.exp(-self.lambda_1 * torch.inner(rv, x)) + 1 * self.lambda_2 * torch.inner(x, x)
            
            
            jacob_h_x = jacobian(cost_func, x_input)

            IF = list([rv[j] - mu[j] for j in range(self.port_num)]) + list([(rv[j] - mu[j])**2 - sigma2[j] for j in range(self.port_num)])
            IF = torch.from_numpy(np.array(IF))
            sample_jacob_h_p = 0
            for j in range(dimx):
                sample_jacob_h_p += jacob_h_x[j] * jacob_x_p[j]
            
            # used for empirical debiasing
            gradient_interact += torch.inner(sample_jacob_h_p, IF) / (loop_num ** 2)

            if type == 'parametric':
                hes_h_x = hessian(cost_func, x_input)
                for j in range(dimx):
                    sample_hes_h_p += (hes_h_x[j][j] * torch.mm(jacob_x_p[j].reshape(-1, 1), jacob_x_p[j].reshape(1, -1)) + jacob_h_x[j] * hes_x_p[j])
                    for k in range(j, dimx):
                        sample_hes_h_p += 2 * hes_h_x[j][k] * torch.mm(jacob_x_p[j].reshape(-1, 1), jacob_x_p[k].reshape(1, -1))

        I_h_p = sample_hes_h_p/loop_num
        # here the esimators we apply are all unbiased, so there is no higher-order bias...

        
        if type == 'parametric':
            return torch.trace(torch.mm(I_h_p, Phi_p)).numpy() / (2 * sample_num)
        elif type == 'empirical':
            return -gradient_interact.numpy()


            

    def saa(self, sample_data):
        weight = cp.Variable(self.port_num)
        sample_num, __ = sample_data.shape
        loss = self.lambda_0 * cp.sum(cp.exp(-self.lambda_1 * sample_data @ weight)) / sample_num + self.lambda_2 * cp.norm(weight) ** 2
        prob = cp.Problem(cp.Minimize(loss))
        prob.solve()
        self.weight = weight.value
        return self.weight

    # we use torch to store the data
    def IO_correction(self, I_h, J_h, sample_num):
        bias_correction = torch.trace(torch.linalg.solve(I_h, J_h))
        return bias_correction.numpy() / sample_num
    
    def saa_correction(self, sol, sample_data):
        sample_num, __ = sample_data.shape
        I_h, J_h = 0, 0
        x_input = torch.from_numpy(sol)
        for i in range(sample_num):
            def cost_func(x):
                sample_torch = torch.from_numpy(sample_data[i])
                return self.lambda_0 * torch.exp(-self.lambda_1 * torch.inner(sample_torch, x)) + self.lambda_2 * torch.inner(x, x)
            
            
            hes_h_x = hessian(cost_func, x_input)
            jacob_h_x = jacobian(cost_func, x_input)

            I_h += hes_h_x 
            J_h += torch.mm(jacob_h_x.reshape(-1, 1), jacob_h_x.reshape(1, -1))
        return self.IO_correction(I_h / sample_num, J_h / sample_num, sample_num)
        

    def uniform_saa(self, sample_data):
        weight_unit = cp.Variable()
        sample_num, __ = sample_data.shape
        loss = self.lambda_0 * cp.sum(cp.exp(-self.lambda_1 * weight_unit * np.sum(sample_data, axis = 1))) / sample_num + self.lambda_2 * self.port_num * cp.square(weight_unit)
        prob = cp.Problem(cp.Minimize(loss))
        prob.solve()
        self.weight = weight_unit.value * np.ones(self.port_num)
        return self.weight

    def uniform_saa_correction(self, sol, sample_data):
        sample_num, __ = sample_data.shape
        I_h, J_h = 0, 0
        x_input = torch.from_numpy(sol[0:1])
        for i in range(sample_num):
            def cost_func(x):
                return self.lambda_0 * torch.exp(-self.lambda_1 * np.sum(sample_data[i]) * x) + self.lambda_2 * self.port_num * torch.square(x)
            
            
            hes_h_x = hessian(cost_func, x_input)
            jacob_h_x = jacobian(cost_func, x_input)

            I_h += hes_h_x 
            J_h += torch.mm(jacob_h_x.reshape(-1, 1), jacob_h_x.reshape(1, -1))
        return self.IO_correction(I_h / sample_num, J_h / sample_num, sample_num)

    def uniform_saa_correction_notorch(self, sol, sample_data):
        sample_num, __ = sample_data.shape
        jacob_h_x, hes_h_x = 0, 0
        sol_val = sol[0]
        for i in range(sample_num):
            exp_term = math.exp(-self.lambda_1 * np.sum(sample_data[i]) * sol_val)
            jacob = 2 * self.lambda_2 * self.port_num * sol_val - self.lambda_0 * self.lambda_1 * np.sum(sample_data[i]) * exp_term
            jacob_h_x += jacob * jacob / sample_num
            hes_h_x += (2 * self.lambda_2 * self.port_num + self.lambda_0 * (self.lambda_1 * np.sum(sample_data[i])) ** 2 * exp_term) / sample_num
        return jacob_h_x / (hes_h_x * sample_num)

    def true_eval(self, sol):
        # mc evaluation, which can be adapted to general mixed parametric setup
        sample_num = 100000
        mc_data = self.generate(sample_num)
        perform_eval = 0
        for i in range(sample_num):
            perform_eval += (self.lambda_0 * math.exp(-self.lambda_1 * np.dot(mc_data[i], sol)) + self.lambda_2 * np.linalg.norm(sol) ** 2) / sample_num
        return perform_eval 
    
    def empirical_eval(self, sol, sample_data):
        sample_num, __ = sample_data.shape
        perform_eval = 0

        for i in range(sample_num):
            perform_eval += (self.lambda_0 * math.exp(-self.lambda_1 * np.dot(sample_data[i], sol)) + self.lambda_2 * np.linalg.norm(sol) ** 2) / sample_num
        return perform_eval         



loop_num = 20
error = np.zeros((8, loop_num))
asset_num = 5

return_mean = np.random.uniform(0, 4, asset_num)
return_cov = np.diag(1 * np.ones(asset_num))
for i in range(asset_num):
    for j in range(i + 1, asset_num):
        rad_val = 0.5 * np.random.rand() 
        return_cov[i][j], return_cov[j][i] = rad_val, rad_val

lambda_0 = 2
lambda_1 = 1
lambda_2 = 3

sample_num = 100
for k in range(loop_num):
    print(k)
    exp_util_model = exp_utility_portfolio(return_mean, return_cov, lambda_0, lambda_1, lambda_2)
    sample_data = exp_util_model.generate(sample_num)


    sol = exp_util_model.saa(sample_data)
    error[0][k] = exp_util_model.true_eval(sol) - exp_util_model.empirical_eval(sol, sample_data)
    error[1][k] = error[0][k] - exp_util_model.saa_correction(sol, sample_data)

    sol = exp_util_model.uniform_saa(sample_data)
    error[2][k] = exp_util_model.true_eval(sol)  - exp_util_model.empirical_eval(sol, sample_data)
    error[3][k] = error[2][k] - exp_util_model.uniform_saa_correction(sol, sample_data)
    error[4][k] = error[2][k] - exp_util_model.uniform_saa_correction_notorch(sol, sample_data)    

    sol_pf, __, __ = exp_util_model.normal_idpd_opt(sample_data)
    error[4][k] = exp_util_model.true_eval(sol_pf) - exp_util_model.normal_pf_eval(sol_pf, sample_data)
    error[5][k] = error[4][k] - exp_util_model.normal_idpd_pf_correction(sol_pf, sample_data, 'parametric')

    error[6][k] = exp_util_model.true_eval(sol_pf) - exp_util_model.empirical_eval(sol_pf, sample_data)
    error[7][k] = error[6][k] - exp_util_model.normal_idpd_pf_correction(sol_pf, sample_data, 'empirical')

#print(exp_util_model.uniform_saa_correction(sol, sample_data))


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


In [3]:
import torch
import numpy as np
for i in range(5):
    print(np.random.rand())

0.8626315886996692
0.7655602322157061
0.5444843042684765
0.4262806110176828
0.2436176226099681


In [18]:
np.mean(error, axis = 1)

array([0.00174144, 0.00093488, 0.00154631, 0.00148233, 0.01664003,
       0.01570379, 0.00180563, 0.00121015])

In [67]:
#implicit function derivative
f_x_p = hes[0][1][index].reshape(1, -1)
f_p_x = f_x_p.reshape(-1, 1)
f_x_x = hes[0][0][index][index]
f_p_p = hes[1][1]

j_x = jacob[0][index]
j_p = jacob[1]

partial_x_p = -j_p / j_x
partial_x2_p2 = -(j_x**2 * f_p_p + f_x_x * torch.mm(j_p.reshape(-1, 1), j_p.reshape(1, -1)) - j_x * (torch.mm(j_p.reshape(-1, 1), f_x_p) + torch.mm(f_p_x, j_p.reshape(1, -1))))/(j_x**3)

In [73]:
dimx = 3
rv = torch.tensor([1, 0.5, 2])
x_input = torch.tensor([0.5, 0.2, 0.4])

loop_num = 11
sample_jacob_h_p = 0 
sample_hes_h_p = 0

for i in range(loop_num):
    rv = torch.normal(mean = mu_input, std = torch.sqrt(sigma2_input))
    
    def cost_func(x):
        return torch.exp(-gamma1 * torch.inner(rv, x)) + gamma2 * torch.inner(x, x)
    
    hes_h_x = hessian(cost_func, x_input)
    jacob_h_x = jacobian(cost_func, x_input)

    for j in range(dimx):
        sample_jacob_h_p += jacob_h_x[j] * jacob_x_p[j]
    
    for j in range(dimx):
        sample_hes_h_p += (hes_h_x[j][j] * torch.mm(jacob_x_p[j].reshape(-1, 1), jacob_x_p[j].reshape(1, -1)) + jacob_h_x[j] * hes_x_p[j])
        for k in range(j, dimx):
            sample_hes_h_p += 2 * hes_h_x[j][k] * torch.mm(jacob_x_p[j].reshape(-1, 1), jacob_x_p[k].reshape(1, -1))

print(sample_jacob_h_p/loop_num)
print(sample_hes_h_p/loop_num)


tensor([ 0.2275, -0.0365,  0.3906, -0.3484, -0.0120, -0.0830])
tensor([[-0.0125, -0.4812, -0.6386, -0.1742,  0.2271,  0.1539],
        [-0.4915,  0.5156, -0.3515,  0.4088, -0.2987,  0.0814],
        [-0.6439, -0.3447,  0.4920,  0.3871,  0.1086, -0.2171],
        [-0.1692,  0.3950,  0.3834,  0.4895, -0.1494, -0.0831],
        [ 0.2335, -0.2995,  0.1125, -0.1575,  0.1792, -0.0230],
        [ 0.1551,  0.0798, -0.2171, -0.0841, -0.0222,  0.0685]])


In [74]:
hes_h_x

tensor([[ 4.0268,  0.1477, -0.0633],
        [ 0.1477,  4.8145, -0.3488],
        [-0.0633, -0.3488,  4.1494]])

In [69]:
partial_x2_p2[0]

tensor([-0.0998, -0.0499, -0.3439,  0.0499,  0.0125,  0.3630])

In [42]:
a = torch.randn(4).reshape(-1, 1)
b = torch.randn(4).reshape(1, -1)
torch.mm(a, b)

tensor([[-0.9488,  0.2638, -0.3576,  1.5430],
        [-0.4789,  0.1332, -0.1805,  0.7788],
        [-0.9093,  0.2528, -0.3427,  1.4788],
        [-2.5149,  0.6993, -0.9479,  4.0899]])